# Using LLM APIs Safely and Reliably

Notebook for **Chapter 5** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 The Generic Request Pattern

In [ ]:
# not run here: shape only. The real function and field names differ per provider.
client = SomeProviderClient(api_key=os.environ["MY_LLM_API_KEY"], timeout=30)
resp = client.chat(
    model="MODEL_NAME_FROM_DOCS",
    messages=[{"role": "system", "content": "You are a triage bot."},
              {"role": "user",   "content": "Summarise: ..."}],
    max_tokens=300, temperature=0)
text  = resp.text
usage = resp.usage      # input_tokens, output_tokens

## 5.2 Retries, Timeouts, Backoff and Streaming

In [ ]:
import time, random, hashlib, os
class TransientError(Exception): pass
class FakeLLM:
    """Pretends to be a provider SDK. Fails the first 2 calls of each prompt."""
    def __init__(self): self.calls={}
    def complete(self, prompt, max_tokens=50):
        n=self.calls.get(prompt,0)+1; self.calls[prompt]=n
        if n<=2: raise TransientError("429 rate limited")
        return {"text":"OK: "+prompt[:20],
                "usage":{"input_tokens":len(prompt.split())*13//10,"output_tokens":12}}
    def stream(self, prompt):
        for w in ("The","answer","is","42."): yield w
def call_with_retry(fn, *a, retries=4, base=0.01, sleep=time.sleep, **kw):
    for attempt in range(retries+1):
        try: return fn(*a, **kw)
        except TransientError as e:
            if attempt==retries: raise
            delay=min(base*2**attempt,1.0)*random.uniform(0.5,1.0)
            print(f"  attempt {attempt+1} failed ({e}); wait {delay*1000:.1f} ms"); sleep(delay)
random.seed(1)
llm=FakeLLM()
r=call_with_retry(llm.complete,"Summarise the weekly sales report for the shop")
print(r)

  attempt 1 failed (429 rate limited); wait 5.7 ms
  attempt 2 failed (429 rate limited); wait 18.5 ms
{'text': 'OK: Summarise the weekly', 'usage': {'input_tokens': 10, 'output_tokens': 12}}


In [ ]:
print("".join(w+" " for w in llm.stream("x")))

The answer is 42. 


## 5.3 Token and Cost Math, Concurrency and Rate Limits

In [ ]:
price_in, price_out = 1.00, 4.00   # made-up, per 1M tokens
reqs=50_000; tin, tout = 1500, 300
print("monthly cost:", reqs*(tin*price_in+tout*price_out)/1e6)
print("with 80% of input cached at 10% price:",
      reqs*(tin*0.2*price_in+tin*0.8*price_in*0.1+tout*price_out)/1e6)
lat=6.0; rps=20; print("concurrent requests in flight:", lat*rps)

monthly cost: 135.0
with 80% of input cached at 10% price: 81.0
concurrent requests in flight: 120.0


## 5.4 Caching and Secrets Handling

In [ ]:
import hashlib
cache={}
def cached(prompt):
    k=hashlib.sha256(prompt.encode()).hexdigest()[:12]
    if k in cache: print("  cache hit",k); return cache[k]
    cache[k]=call_with_retry(llm.complete,prompt); return cache[k]
cached("hello world again"); cached("hello world again")

  attempt 1 failed (429 rate limited); wait 8.8 ms
  attempt 2 failed (429 rate limited); wait 12.6 ms
  cache hit cbb2d50bd9c8


In [ ]:
import os
key=os.environ.get("MY_LLM_API_KEY")
print("key present?", key is not None)

key present? False


## 5.5 Start From Zero: What Really Happens in One API Call

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
class ApiError(Exception):
    def __init__(self, status, message="", retry_after=None):
        super().__init__(f"{status} {message}".strip())
        self.status = status
        self.retry_after = retry_after

def classify(err):
    """Return 'retry' or 'fail' for an exception."""
    if isinstance(err, (TimeoutError, ConnectionError)):
        return "retry"
    if isinstance(err, ApiError):
        if err.status in (408, 429) or err.status >= 500:
            return "retry"
        return "fail"
    return "fail"          # unknown errors are probably our own bug

def worst_case_seconds(attempt_timeout, retries, base, cap=30.0):
    waits = [min(base * 2 ** i, cap) for i in range(retries)]
    return (retries + 1) * attempt_timeout + sum(waits), waits

cases = [ApiError(400, "bad request"), ApiError(401, "bad key"), ApiError(404, "no such model"),
         ApiError(408, "request timeout"), ApiError(413, "payload too large"),
         ApiError(429, "slow down", retry_after=2), ApiError(500, "oops"), ApiError(503, "overloaded"),
         TimeoutError("read timed out"), ConnectionError("reset by peer"), ValueError("my own bug")]
for e in cases:
    print(f"{type(e).__name__:15s} {str(e):22s} -> {classify(e)}")
print(worst_case_seconds(6, 2, 0.5))
print(worst_case_seconds(10, 4, 1))

ApiError        400 bad request        -> fail
ApiError        401 bad key            -> fail
ApiError        404 no such model      -> fail
ApiError        408 request timeout    -> retry
ApiError        413 payload too large  -> fail
ApiError        429 slow down          -> retry
ApiError        500 oops               -> retry
ApiError        503 overloaded         -> retry
TimeoutError    read timed out         -> retry
ConnectionError reset by peer          -> retry
ValueError      my own bug             -> fail
(19.5, [0.5, 1.0])
(65, [1, 2, 4, 8])


## 5.6 Conversation History Done Properly

In [ ]:
SYS, USER, REPLY = 200, 50, 150

def tokens_full(k):
    return SYS + (k - 1) * (USER + REPLY) + USER

def tokens_window(k, keep=4):
    return SYS + min(k - 1, keep) * (USER + REPLY) + USER

def tokens_summary(k, keep=4, summary=150):
    older = k - 1 - keep                       # turns that no longer fit the window
    return SYS + (summary if older > 0 else 0) + min(k - 1, keep) * (USER + REPLY) + USER

print("turn  full  window  summary")
for k in (1, 5, 6, 10, 20):
    print(f"{k:4d} {tokens_full(k):5d} {tokens_window(k):7d} {tokens_summary(k):8d}")
for name, f in (("full", tokens_full), ("window", tokens_window), ("summary", tokens_summary)):
    print(name, "total over 20 turns:", sum(f(k) for k in range(1, 21)))

turn  full  window  summary
   1   250     250      250
   5  1050    1050     1050
   6  1250    1050     1200
  10  2050    1050     1200
  20  4050    1050     1200
full total over 20 turns: 43000
window total over 20 turns: 19000
summary total over 20 turns: 21250


In [ ]:
def est_tokens(text):
    return int(len(text.split()) * 1.3) + 4          # +4 for per-message overhead

def trim_to_budget(messages, budget):
    system, rest = messages[0], messages[1:]
    kept, used = [], est_tokens(system["content"])
    for m in reversed(rest):                          # newest first
        cost = est_tokens(m["content"])
        if used + cost > budget:
            break
        kept.append(m); used += cost
    return [system] + kept[::-1], used

msgs = [{"role": "system", "content": "You are a support bot for a small shop."}]
for i in range(1, 9):
    msgs.append({"role": "user", "content": f"Question number {i} " + "about my order " * 5})
    msgs.append({"role": "assistant", "content": f"Answer number {i} " + "here are the details " * 6})
trimmed, used = trim_to_budget(msgs, 150)
print("total messages:", len(msgs), "kept:", len(trimmed), "tokens used:", used)
print([m["content"][:10] for m in trimmed])

total messages: 17 kept: 5 tokens used: 147
['You are a ', 'Question n', 'Answer num', 'Question n', 'Answer num']


## 5.7 Retries in Depth: Jitter, Deadlines, Amplification and Circuit Breakers

In [ ]:
import random
random.seed(7)
N, BASE, CAP = 300, 1.0, 30.0

def ceiling(attempt):
    return min(BASE * 2 ** attempt, CAP)

strategies = {
    "no jitter":     lambda a: ceiling(a),
    "random(.5,1)":  lambda a: ceiling(a) * random.uniform(0.5, 1.0),
    "full jitter":   lambda a: random.uniform(0, ceiling(a)),
    "equal jitter":  lambda a: ceiling(a) / 2 + random.uniform(0, ceiling(a) / 2),
}
for name, wait in strategies.items():
    times = []
    for _ in range(N):
        t = 0.0
        for a in range(3):                 # three retries per client
            t += wait(a)
            times.append(t)
    buckets = {}
    for t in times:
        b = int(t * 10)                    # 100 ms bucket number
        buckets[b] = buckets.get(b, 0) + 1
    print(f"{name:13s} busiest 100 ms window: {max(buckets.values()):4d} retries")

no jitter     busiest 100 ms window:  300 retries
random(.5,1)  busiest 100 ms window:   63 retries
full jitter   busiest 100 ms window:   47 retries
equal jitter  busiest 100 ms window:   72 retries


In [ ]:
import random

class Clock:
    def __init__(self): self.t = 0.0
    def sleep(self, s): self.t += s

class ApiError(Exception):
    def __init__(self, status, retry_after=None):
        super().__init__(str(status)); self.status = status; self.retry_after = retry_after

class DeadlineExceeded(Exception): pass

def is_retryable(e):
    if isinstance(e, (TimeoutError, ConnectionError)): return True
    return isinstance(e, ApiError) and (e.status in (408, 429) or e.status >= 500)

def call_with_policy(fn, clock, max_retries=4, base=0.5, cap=8.0, deadline=20.0, rng=random):
    start, attempts = clock.t, 0
    while True:
        attempts += 1
        try:
            return fn(), attempts, round(clock.t - start, 2)
        except Exception as e:
            if not is_retryable(e) or attempts > max_retries:
                raise
            wait = getattr(e, "retry_after", None)
            if wait is None:
                wait = rng.uniform(0, min(cap, base * 2 ** (attempts - 1)))
            if clock.t - start + wait >= deadline:
                raise DeadlineExceeded(f"gave up after {attempts} attempts, {clock.t - start:.1f}s")
            clock.sleep(wait)

def make_fn(clock, script, latency=0.8):
    it = iter(script)
    def fn():
        clock.sleep(latency)               # every attempt takes 0.8 s
        outcome = next(it)
        if isinstance(outcome, Exception): raise outcome
        return outcome
    return fn

random.seed(3)
def run(label, script, **kw):
    clock = Clock()
    try:
        print(label, "->", call_with_policy(make_fn(clock, script), clock, **kw))
    except Exception as e:
        print(label, "->", type(e).__name__, e, "| clock", round(clock.t, 2))

run("429 then 503 then ok", [ApiError(429, retry_after=2), ApiError(503), "ok"])
run("400 bad request     ", [ApiError(400)])
run("503 forever, 5 s    ", [ApiError(503)] * 10, deadline=5)
run("timeouts, 2 retries ", [TimeoutError("slow")] * 5, max_retries=2)

429 then 503 then ok -> ('ok', 3, 4.64)
400 bad request      -> ApiError 400 | clock 0.8
503 forever, 5 s     -> DeadlineExceeded gave up after 4 attempts, 5.0s | clock 5.05
timeouts, 2 retries  -> TimeoutError slow | clock 2.45


In [ ]:
class Breaker:
    def __init__(self, threshold=3, cooldown=10.0):
        self.threshold, self.cooldown = threshold, cooldown
        self.fails, self.opened_at, self.state = 0, None, "closed"
    def allow(self, now):
        if self.state == "open":
            if now - self.opened_at >= self.cooldown:
                self.state = "half-open"
                return True
            return False
        return True
    def record(self, ok, now):
        if ok:
            self.fails, self.state = 0, "closed"
            return
        self.fails += 1
        if self.state == "half-open" or self.fails >= self.threshold:
            self.state, self.opened_at = "open", now

b = Breaker()
for now in range(0, 41, 4):                    # one call every 4 s
    if not b.allow(now):
        print(f"t={now:2d}s  blocked (breaker open)")
        continue
    ok = now >= 25                             # provider recovers at t=25
    b.record(ok, now)
    print(f"t={now:2d}s  call {'ok  ' if ok else 'FAIL'} -> state {b.state}")

t= 0s  call FAIL -> state closed
t= 4s  call FAIL -> state closed
t= 8s  call FAIL -> state open
t=12s  blocked (breaker open)
t=16s  blocked (breaker open)
t=20s  call FAIL -> state open
t=24s  blocked (breaker open)
t=28s  blocked (breaker open)
t=32s  call ok   -> state closed
t=36s  call ok   -> state closed
t=40s  call ok   -> state closed


## 5.8 Timeouts, Streaming and Cancellation

In [ ]:
import json
raw = ('data: {"delta": "The"}\n\n'
       'data: {"delta": " answer"}\n\n'
       'data: {"delta": " is 42."}\n\n'
       'data: {"usage": {"input_tokens": 12, "output_tokens": 5}}\n\n'
       'data: [DONE]\n\n')

def parse_sse(raw_text):
    text, usage = "", None
    for block in raw_text.split("\n\n"):
        for line in block.strip().splitlines():
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                return text, usage, True
            event = json.loads(data)
            text += event.get("delta", "")
            usage = event.get("usage", usage)
    return text, usage, False

print(parse_sse(raw))
cut = "\n\n".join(raw.split("\n\n")[:2]) + "\n\n"       # connection died after 2 events
print(parse_sse(cut))

('The answer is 42.', {'input_tokens': 12, 'output_tokens': 5}, True)
('The answer', None, False)


In [ ]:
def consume(stream, first_token_limit=6.0, max_gap=2.0):
    text, first, last = [], None, 0.0
    for t, tok in stream:
        limit = first_token_limit if first is None else max_gap
        if t - last > limit:
            return {"status": "stalled", "partial": "".join(text), "silent_for_s": round(t - last, 2)}
        if first is None:
            first = t
        text.append(tok); last = t
    n = len(text)
    return {"status": "ok", "ttft_s": first, "total_s": last,
            "tokens_per_s": round((n - 1) / (last - first), 1), "text": "".join(text)}

healthy = [(0.8 + 0.1 * i, w) for i, w in enumerate(["Retry ", "only ", "transient ", "errors ", "with ", "jitter."])]
stalled = healthy[:3] + [(9.0, "never shown")]
print(consume(healthy))
print(consume(stalled))

{'status': 'ok', 'ttft_s': 0.8, 'total_s': 1.3, 'tokens_per_s': 10.0, 'text': 'Retry only transient errors with jitter.'}
{'status': 'stalled', 'partial': 'Retry only transient ', 'silent_for_s': 8.0}


## 5.9 Cost, Capacity and Rate Limits: Solving the Numbers Properly

In [ ]:
P_IN, P_OUT = 1.0, 4.0                      # made-up, per million tokens

def call_cost(tin, tout, cached_frac=0.0, cache_price=0.1):
    inp = tin * (1 - cached_frac) * P_IN + tin * cached_frac * P_IN * cache_price
    return (inp + tout * P_OUT) / 1e6

base = call_cost(1500, 300)
extra = 0.04 + 0.02                         # 4% truncated and rerun, 2% bad JSON and repaired
per_task = base * (1 + extra)
success = 0.97                              # 3% of tasks never succeed but still cost money
print(f"per call {base:.6f}   per task {per_task:.6f}   per SUCCESSFUL task {per_task / success:.6f}")
print(f"monthly, 50,000 tasks: naive {50_000 * base:.2f}   real {50_000 * per_task / success:.2f}")

per call 0.002700   per task 0.002862   per SUCCESSFUL task 0.002951
monthly, 50,000 tasks: naive 135.00   real 147.53


In [ ]:
P_IN, P_OUT = 1.0, 4.0
scenarios = [("baseline", 1500, 300, 0.0, 0.10),
             ("80% prefix cache", 1500, 300, 0.8, 0.10),
             ("+ output cap 150", 1500, 150, 0.8, 0.10),
             ("+ 1% wasted calls", 1500, 150, 0.8, 0.01)]
for name, tin, tout, cached, waste in scenarios:
    inp = tin * (1 - cached) * P_IN + tin * cached * P_IN * 0.1
    per_call = inp + tout * P_OUT
    print(f"{name:18s} cost per 1,000 requests: {per_call * (1 + waste) / 1e3:.3f}")

baseline           cost per 1,000 requests: 2.970
80% prefix cache   cost per 1,000 requests: 1.782
+ output cap 150   cost per 1,000 requests: 1.122
+ 1% wasted calls  cost per 1,000 requests: 1.030


In [ ]:
def max_sustainable(rpm_limit, tpm_limit, tokens_per_request):
    by_tokens = tpm_limit / tokens_per_request
    binding = "requests" if rpm_limit <= by_tokens else "tokens"
    return round(min(rpm_limit, by_tokens), 1), binding

print(max_sustainable(500, 200_000, 2_400))     # long prompts
print(max_sustainable(500, 200_000, 300))       # short prompts

(83.3, 'tokens')
(500, 'requests')


In [ ]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate, self.cap = rate, capacity
        self.level, self.t = capacity, 0.0          # starts full
    def take(self, n, now):
        self.level = min(self.cap, self.level + (now - self.t) * self.rate)
        self.t = now
        if self.level >= n:
            self.level -= n
            return 0.0                              # go now
        return round((n - self.level) / self.rate, 2)   # seconds to wait

bucket = TokenBucket(rate=1000, capacity=10_000)    # 60,000 tokens per minute
for t in range(12):                                 # a 2,500-token request every second
    wait = bucket.take(2500, t)
    print(f"t={t:2d}s  " + ("sent, bucket left %5d" % bucket.level if wait == 0 else f"must wait {wait} s"))

t= 0s  sent, bucket left  7500
t= 1s  sent, bucket left  6000
t= 2s  sent, bucket left  4500
t= 3s  sent, bucket left  3000
t= 4s  sent, bucket left  1500
t= 5s  sent, bucket left     0
t= 6s  must wait 1.5 s
t= 7s  must wait 0.5 s
t= 8s  sent, bucket left   500
t= 9s  must wait 1.0 s
t=10s  sent, bucket left     0
t=11s  must wait 1.5 s


In [ ]:
def plan(rate, latency, limit):
    inflight = rate * latency
    verdict = "OK" if inflight <= limit else f"over by {inflight - limit:.0f}"
    return inflight, verdict, round(limit / latency, 1)

for rate, latency in ((20, 6), (20, 4), (50, 3), (5, 12)):
    print(rate, "req/s,", latency, "s ->", plan(rate, latency, limit=100))

20 req/s, 6 s -> (120, 'over by 20', 16.7)
20 req/s, 4 s -> (80, 'OK', 25.0)
50 req/s, 3 s -> (150, 'over by 50', 33.3)
5 req/s, 12 s -> (60, 'OK', 8.3)


## 5.10 Caching Done Right: Keys, Expiry, Similarity and Prefixes

In [ ]:
import hashlib, json

def cache_key(model, params, system, messages, tenant):
    payload = {"model": model, "params": params, "system": system,
               "messages": messages, "tenant": tenant}
    blob = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]

class TTLCache:
    def __init__(self, ttl):
        self.ttl, self.data, self.hits, self.misses = ttl, {}, 0, 0
    def get(self, key, now):
        item = self.data.get(key)
        if item and now - item[1] < self.ttl:
            self.hits += 1
            return item[0]
        self.misses += 1
        return None
    def put(self, key, value, now):
        self.data[key] = (value, now)

msgs = [{"role": "user", "content": "Where is my order?"}]
k1 = cache_key("m1", {"temperature": 0, "max_tokens": 200}, "Be brief.", msgs, "shopA")
k2 = cache_key("m1", {"max_tokens": 200, "temperature": 0}, "Be brief.", msgs, "shopA")   # dict order differs
k3 = cache_key("m1", {"temperature": 0, "max_tokens": 200}, "Be brief.", msgs, "shopB")   # other tenant
k4 = cache_key("m1", {"temperature": 0.7, "max_tokens": 200}, "Be brief.", msgs, "shopA") # other sampling
print(k1 == k2, k1 == k3, k1 == k4)

c = TTLCache(ttl=60)
c.put(k1, "Your order ships Friday.", now=0)
print(c.get(k1, now=30), "|", c.get(k1, now=61), "|", c.get(k3, now=30))
print("hits", c.hits, "misses", c.misses)

True False False
Your order ships Friday. | None | None
hits 1 misses 2


In [ ]:
import numpy as np
def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

stored = ("How do I cancel my order?", np.array([0.90, 0.30, 0.10]))
new = {"How can I cancel my order": np.array([0.88, 0.33, 0.10]),          # same meaning
       "How do I cancel my subscription?": np.array([0.85, 0.45, 0.15]),   # different action
       "What is your refund time?": np.array([0.20, 0.90, 0.30])}          # different topic
for q, v in new.items():
    s = cos(stored[1], v)
    hits = [t for t in (0.95, 0.99, 0.999) if s >= t]
    print(f"{s:.4f}  hit at thresholds {hits}  <- {q}")

0.9993  hit at thresholds [0.95, 0.99, 0.999]  <- How can I cancel my order
0.9854  hit at thresholds [0.95]  <- How do I cancel my subscription?
0.5190  hit at thresholds []  <- What is your refund time?


In [ ]:
TIN, PREFIX = 3000, 2400
WRITE, READ = 1.25, 0.10                    # hypothetical multipliers
print("calls  no_cache  with_cache  saved")
for n in range(1, 6):
    no_cache = n * TIN
    first = PREFIX * WRITE + (TIN - PREFIX)
    later = PREFIX * READ + (TIN - PREFIX)
    with_cache = first + (n - 1) * later
    print(f"{n:5d} {no_cache:9.0f} {with_cache:11.0f} {no_cache - with_cache:6.0f}")

calls  no_cache  with_cache  saved
    1      3000        3600   -600
    2      6000        4440   1560
    3      9000        5280   3720
    4     12000        6120   5880
    5     15000        6960   8040


## 5.11 Secrets, Logs and Privacy in Practice

In [ ]:
import os, re

def get_key(name="MY_LLM_API_KEY"):
    key = os.environ.get(name)
    if not key:
        raise RuntimeError(f"{name} is not set. Export it in your shell or secret manager; never hard-code it.")
    return key

try:
    get_key()
except RuntimeError as e:
    print("startup check:", e)

PATTERNS = {
    "api_key": re.compile(r"\b(?:sk|key|tok)[-_][A-Za-z0-9]{16,}\b"),
    "email":   re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b"),
    "phone":   re.compile(r"\b[6-9]\d{9}\b"),
}

def redact(text):
    for name, pattern in PATTERNS.items():
        text = pattern.sub(f"[{name.upper()}]", text)
    return text

sample = "User asha@example.com (9876543210) pasted key sk-FAKE1234567890ABCDEFGH and asked about order 4471."
print(redact(sample))

startup check: MY_LLM_API_KEY is not set. Export it in your shell or secret manager; never hard-code it.
User [EMAIL] ([PHONE]) pasted key [API_KEY] and asked about order 4471.


In [ ]:
import re
RULES = [("looks like an API key", re.compile(r"\b(?:sk|key|tok)[-_][A-Za-z0-9]{16,}\b")),
         ("hard-coded credential", re.compile(r"(?i)\b(?:password|secret|token|api_key)\b\s*[:=]\s*[\"']?(?!os\.environ)[^\s\"']{8,}"))]

def scan(diff_text):
    findings, fname = [], "?"
    for i, line in enumerate(diff_text.splitlines(), 1):
        if line.startswith("+++ "):
            fname = line[4:]
            continue
        for why, pattern in RULES:
            if pattern.search(line):
                findings.append((fname, i, why))
    return findings

diff = '''+++ notebook.py
+import os
+API_KEY = "sk-live9f8e7d6c5b4a39281716"
+client = Client(api_key=os.environ["MY_LLM_API_KEY"])
+++ config.yaml
+password: hunter2hunter2
'''
for f in scan(diff):
    print(f)

('notebook.py', 3, 'looks like an API key')
('notebook.py', 3, 'hard-coded credential')
('config.yaml', 6, 'hard-coded credential')


## 5.12 Reliable Structured Output and Tool Calls

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import json

class FakeModel:
    def __init__(self, replies):
        self.replies, self.calls = list(replies), 0
    def complete(self, messages):
        self.calls += 1
        return {"text": self.replies.pop(0), "stop_reason": "end_turn"}

def validate(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as e:
        return None, f"not valid JSON: {e.msg} at character {e.pos}"
    if not isinstance(obj, dict):
        return None, "top level must be an object"
    if not isinstance(obj.get("category"), str):
        return None, "field 'category' must be a string"
    if type(obj.get("priority")) is not int:
        return None, "field 'priority' must be an integer"
    if obj["priority"] not in (1, 2, 3):
        return None, "priority must be 1, 2 or 3"
    return obj, None

def get_structured(model, prompt, max_repairs=2):
    messages = [{"role": "user", "content": prompt}]
    error = None
    for attempt in range(1, max_repairs + 2):
        reply = model.complete(messages)
        if reply["stop_reason"] == "max_tokens":
            obj, error = None, "output was cut off at the token limit"
        else:
            obj, error = validate(reply["text"])
        if obj is not None:
            return obj, attempt
        messages += [{"role": "assistant", "content": reply["text"]},
                     {"role": "user", "content": f"Invalid: {error}. Reply with corrected JSON only."}]
    raise ValueError(f"no valid output after {max_repairs + 1} tries; last error: {error}")

good_in_the_end = FakeModel(['Sure! {"category": "billing"}',
                             '{"category": "billing", "priority": "high"}',
                             '{"category": "billing", "priority": 2}'])
print(get_structured(good_in_the_end, "Classify this ticket as JSON."))
never_good = FakeModel(["I cannot do that."] * 3)
try:
    get_structured(never_good, "Classify this ticket as JSON.")
except ValueError as e:
    print("ValueError:", e)

({'category': 'billing', 'priority': 2}, 3)
ValueError: no valid output after 3 tries; last error: not valid JSON: Expecting value at character 0


In [ ]:
class EmailTool:
    def __init__(self):
        self.sent, self.done = [], {}
    def send(self, to, body, idem_key=None):
        if idem_key and idem_key in self.done:
            return self.done[idem_key]                  # replay, no second email
        self.sent.append((to, body))
        result = f"sent#{len(self.sent)}"
        if idem_key:
            self.done[idem_key] = result
        return result

unsafe, safe = EmailTool(), EmailTool()
for _ in range(2):                                      # first try "timed out", client retried
    unsafe.send("asha@example.com", "Refund approved")
print("without key, emails sent:", len(unsafe.sent))
r1 = safe.send("asha@example.com", "Refund approved", "refund-8812-email")
r2 = safe.send("asha@example.com", "Refund approved", "refund-8812-email")
r3 = safe.send("ravi@example.com", "Refund approved", "refund-9000-email")
print("with key:", r1, r2, r3, "emails sent:", len(safe.sent))

without key, emails sent: 2
with key: sent#1 sent#1 sent#2 emails sent: 2


## 5.13 How to Solve Any Using LLM APIs Safely and Reliably Problem

Every problem in this topic, from an interview question to a 2 a.m. incident, has the same shape: something is flaky, slow, expensive, throttled or unsafe, and you must find the cause and fix it with numbers to back you up. This section is a playbook. Read it once, then use it as a checklist whenever you are stuck.

In [ ]:
def chatbot_cost(turns=6, sys_=300, user=60, reply=180, keep=None, p_in=1.0, p_out=4.0):
    total_in = 0
    for k in range(1, turns + 1):
        older = k - 1 if keep is None else min(k - 1, keep)
        total_in += sys_ + older * (user + reply) + user
    total_out = turns * reply
    return (total_in * p_in + total_out * p_out) / 1e6

monthly = 2000 * 30
base = chatbot_cost()
print("D baseline monthly:", round(base * monthly, 1))
for label, kw in (("window 2", dict(keep=2)), ("reply 120", dict(reply=120)), ("both", dict(keep=2, reply=120))):
    m = chatbot_cost(**kw) * monthly
    print(f"E {label:9s} monthly {m:6.1f}  saving {100 * (1 - m / (base * monthly)):4.1f}%")
print("B max tokens:", (8 - 0.5 - 0.8) * 50, " total with 300:", 0.5 + 0.8 + 300 / 50)
inflight, capacity = 30 * 5, 100 / 5
backlog = (30 - capacity) * 60
print("F in flight", inflight, "capacity/s", capacity, "backlog after 60 s", backlog, "wait", backlog / capacity)

D baseline monthly: 604.8
E window 2  monthly  518.4  saving 14.3%
E reply 120 monthly  464.4  saving 23.2%
E both      monthly  399.6  saving 33.9%
B max tokens: 335.0  total with 300: 7.3
F in flight 150 capacity/s 20.0 backlog after 60 s 600.0 wait 30.0


In [ ]:
import pandas as pd, numpy as np
rows = [
 (200, 2.1, 1500, 300, "miss", "end_turn"),  (200, 0.0, 0, 0, "hit", "end_turn"),
 (200, 2.4, 1500, 310, "miss", "end_turn"),  (429, 0.3, 0, 0, "miss", "none"),
 (200, 3.9, 1500, 300, "miss", "end_turn"),  (200, 6.2, 1500, 512, "miss", "max_tokens"),
 (503, 1.0, 0, 0, "miss", "none"),           (200, 2.2, 1500, 290, "miss", "end_turn"),
 (200, 0.0, 0, 0, "hit", "end_turn"),        (400, 0.2, 0, 0, "miss", "none"),
 (200, 5.8, 1500, 512, "miss", "max_tokens"), (200, 2.0, 1500, 305, "miss", "end_turn"),
]
df = pd.DataFrame(rows, columns=["status", "latency_s", "tin", "tout", "cache", "stop"])
ok = df[df.status == 200]
useful = ok[ok.stop == "end_turn"]
real = ok[ok.cache == "miss"]
cost = (df.tin * 1.0 + df.tout * 4.0).sum() / 1e6
print("status mix:", df.status.value_counts().sort_index().to_dict())
print("http ok:", len(ok), "of", len(df), "| useful (not truncated):", len(useful))
print("truncated share of ok calls:", round((ok.stop == "max_tokens").mean(), 3))
print("cache hit share:", round((df.cache == "hit").mean(), 3))
print("latency p50 / p95 of real successful calls:", np.percentile(real.latency_s, [50, 95]).round(2).tolist())
print("total cost:", round(cost, 6), "| cost per useful task:", round(cost / len(useful), 6))

status mix: {200: 9, 400: 1, 429: 1, 503: 1}
http ok: 9 of 12 | useful (not truncated): 7
truncated share of ok calls: 0.222
cache hit share: 0.167
latency p50 / p95 of real successful calls: [2.4, 6.08]
total cost: 0.020616 | cost per useful task: 0.002945


## 5.14 Going Deeper: Queues, Tail Latency, Delivery Guarantees and What Interviewers Probe

In [ ]:
service = 2.0                                  # seconds of work per request
for rho in (0.5, 0.8, 0.9, 0.95, 0.99):
    time_in_system = service / (1 - rho)
    print(f"utilisation {rho:4.0%}  average time in system {time_in_system:6.1f} s  ({time_in_system / service:5.1f}x the work)")

utilisation  50%  average time in system    4.0 s  (  2.0x the work)
utilisation  80%  average time in system   10.0 s  (  5.0x the work)
utilisation  90%  average time in system   20.0 s  ( 10.0x the work)
utilisation  95%  average time in system   40.0 s  ( 20.0x the work)
utilisation  99%  average time in system  200.0 s  (100.0x the work)


In [ ]:
import numpy as np
rng = np.random.default_rng(11)
n = 100_000
first = rng.lognormal(mean=np.log(2.0), sigma=0.6, size=n)     # seconds, median 2, heavy tail
second = rng.lognormal(mean=np.log(2.0), sigma=0.6, size=n)    # the hedge copy
trigger = np.percentile(first, 95)
hedged = np.where(first > trigger, np.minimum(first, trigger + second), first)

def pct(x):
    return np.percentile(x, [50, 95, 99]).round(2).tolist()

print("trigger at p95 =", round(float(trigger), 2), "s")
print("plain  p50/p95/p99:", pct(first))
print("hedged p50/p95/p99:", pct(hedged))
print("extra requests sent:", round(float((first > trigger).mean()) * 100, 1), "%")

trigger at p95 = 5.35 s
plain  p50/p95/p99: [2.0, 5.35, 8.0]
hedged p50/p95/p99: [2.0, 5.35, 7.08]
extra requests sent: 5.0 %


## 5.15 Mixed-Practice Drill

These twelve problems mix every pattern, in no particular order, as in a real interview or incident. Solve each on paper first, then read the answer. If you miss one, the section number tells you where to go back.

## 5.16 Self-Test, Scoring Guide and What Comes Next

Close the chapter and answer all twelve from memory, with a pencil. One point each. Then check the answers below.

## Chapter recap

## Practice set

Problem 5.P.1 (easy) Which errors do you retry: 400, 429, 500, 403? **Answer:** 429 and 500 only.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch05.html